# EM spectrometer example

This notebook builds a simple magnetic spectrometer, tracks an electron ensemble, and plots the energy calibration and dispersion at several detector screens. The field below is an idealized hard-edged model for demonstrating the package workflow; use a measured or simulated field map for quantitative planning.

The package expects positions in metres, time in seconds, electric field in V/m, magnetic field in tesla, and momentum normalized to the particle mass times c. Install the package and its dependencies before running the notebook.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.constants import c, e, m_e
from scipy.interpolate import PchipInterpolator

from EMspecPy.spectrometer import EMspec


## Define the magnetic field

Coordinates are chosen as follows:

- x is the initial beam direction.
- y is the magnetic deflection and dispersion direction.
- z is the main magnetic-field direction.

The callback accepts a time and a 3 by N position array, then returns electric and magnetic field arrays with the same shape. This example uses a static, uniform field inside a rectangular magnet gap and zero electric field.


In [ ]:
def em_field(time_s, positions_m):
    electric_field = np.zeros_like(positions_m)
    magnetic_field = np.zeros_like(positions_m)

    magnet_start_x = 0.20
    magnet_length_x = 0.30
    magnet_center_y = 0.0
    magnet_half_gap_y = 0.06
    field_t = 0.83

    inside_magnet = (
        (positions_m[0] >= magnet_start_x)
        & (positions_m[0] <= magnet_start_x + magnet_length_x)
        & (np.abs(positions_m[1] - magnet_center_y) <= magnet_half_gap_y)
    )
    magnetic_field[2, inside_magnet] = field_t
    return electric_field, magnetic_field


### Inspect the field in the x-y plane

The field is evaluated in a vectorized call and reshaped to the plotting grid. The notebook plots Bz because it is the component that bends the beam in this coordinate convention.


In [ ]:
x_m = np.linspace(-0.10, 0.80, 500)
y_m = np.linspace(-0.20, 0.20, 250)
X_m, Y_m = np.meshgrid(x_m, y_m, indexing="xy")
positions_m = np.vstack((X_m.ravel(), Y_m.ravel(), np.zeros(X_m.size)))
_, magnetic_field = em_field(0.0, positions_m)
Bz_t = magnetic_field[2].reshape(Y_m.shape)

fig, ax = plt.subplots(figsize=(10, 4))
image = ax.pcolormesh(x_m, y_m, Bz_t, cmap="RdBu_r", vmin=-0.85, vmax=0.85, shading="auto")
fig.colorbar(image, ax=ax, label=r"$B_z$ [T]")
ax.set(xlabel="x [m]", ylabel="y [m]", title="Idealized magnet field")
plt.show()


## Define detector screens

A screen is a finite rectangle defined by three 3D points: its origin, the far end of its length, and a point across its width. The helper below takes the screen center, length, width, and angle in the x-y plane, then places the origin at the lower-z corner so the screen is centered on z=0.


In [ ]:
def make_screen(label, center_m, length_m, width_m, angle_rad):
    center_m = np.asarray(center_m, dtype=float)
    along = np.array([np.cos(angle_rad), np.sin(angle_rad), 0.0]) * length_m
    across = np.array([0.0, 0.0, width_m])
    origin = center_m - across / 2
    return {
        "label": label,
        "origin": origin,
        "end": origin + along,
        "side": origin + across,
    }

screen_length_m = 0.30
screen_width_m = 0.075
screens = [
    make_screen("Near magnet", (0.35, 0.06, 0.0), screen_length_m, screen_width_m, np.deg2rad(0)),
    make_screen("Intermediate 1", (1.05, 0.00, 0.0), screen_length_m, screen_width_m, np.deg2rad(135)),
    make_screen("Intermediate 2", (2.00, 0.00, 0.0), screen_length_m, screen_width_m, np.deg2rad(135)),
    make_screen("Exit", (3.38, 0.00, 0.0), screen_length_m, screen_width_m, np.deg2rad(90)),
]


## Initialize and track electrons

The fan is placed in the z direction, perpendicular to the x-y dispersion plane. In this package, phi=0 gives divergence in z. The energy distribution is specified by Lorentz factor gamma, not kinetic energy; gamma values of 100 to 4000 correspond to roughly 50.6 MeV to 2.04 GeV for electrons.

The spatial limits bound the tracking region. The dx value sets the nominal maximum distance per step from the initial maximum speed; reduce it when the field varies sharply.


In [ ]:
n_energies = 31
n_fan_angles = 5
gamma_limits = (100.0, 4000.0)
divergence_rad = 2e-3
step_distance_m = 0.1e-3
max_steps = 100_000
position_limits_m = [[-0.1, 3.5], [-0.5, 0.5], [-0.5, 0.5]]
momentum_limits = [[None, None], [None, None], [None, None]]

spectrometer = EMspec(em_field)
initial_state = spectrometer.initialise_particles(
    N_g=n_energies,
    N_t=n_fan_angles,
    dist_type="fan",
    g_lims=gamma_limits,
    div_rad=divergence_rad,
    phi=0.0,
)
print(f"Initialized {initial_state.shape[1]} particles")


In [ ]:
spectrometer.model_spectrometer(
    screens=screens,
    dx=step_distance_m,
    xLims=position_limits_m,
    pLims=momentum_limits,
    N_max=max_steps,
    calc_dispersion=True,
    show_tracks=False,
)

# Plot trajectories separately from calibration plots.
ax, colorbar = spectrometer.plot_tracks(plot_Bfield=True)
ax.set_title("Electron trajectories and magnet field")
plt.show()


## Plot screen position against energy

Each energy has several fan trajectories. The plot shows their mean impact coordinate and the min-to-max range. Screen positions are measured along each screen's length from its origin.


In [ ]:
def screen_energy_summary(screen):
    energy_mev = np.asarray(screen["screen_E"])
    position_m = np.asarray(screen["screen_x"])
    valid = np.isfinite(energy_mev) & np.isfinite(position_m)
    energies = np.unique(energy_mev[valid])
    mean_x, min_x, max_x = [], [], []
    for energy in energies:
        x_at_energy = position_m[valid & (energy_mev == energy)]
        mean_x.append(np.mean(x_at_energy))
        min_x.append(np.min(x_at_energy))
        max_x.append(np.max(x_at_energy))
    return energies, np.asarray(mean_x), np.asarray(min_x), np.asarray(max_x)

fig, ax = plt.subplots(figsize=(8, 5))
for screen in screens:
    energy, mean_x, min_x, max_x = screen_energy_summary(screen)
    if energy.size == 0:
        continue
    line, = ax.plot(energy, mean_x, label=screen["label"])
    ax.fill_between(energy, min_x, max_x, color=line.get_color(), alpha=0.2)
ax.set(xlabel="Kinetic energy [MeV]", ylabel="Screen position [m]", title="Screen response")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()


## Query the mean position at a chosen energy

The screen record contains one value per particle, so interpolate after grouping particles by energy. This avoids passing repeated energy values directly to an interpolator. The query returns NaN outside the simulated energy range.


In [ ]:
screen = screens[1]
energy_mev, mean_x_m, _, _ = screen_energy_summary(screen)
if energy_mev.size >= 2:
    energy_to_position = PchipInterpolator(energy_mev, mean_x_m, extrapolate=False)
    query_energy_mev = 1000.0
    print(f"Mean position at {query_energy_mev:.0f} MeV: {energy_to_position(query_energy_mev):.4f} m")
else:
    print(f"{screen['label']}: not enough energy points for an interpolation")


## Inspect the calculated dispersion

The model stores energy as a function of screen position and its derivative in MeV/mm. A screen with too few particles or energy points has NaN calibration arrays and emits a warning.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for screen in screens:
    if "spec_MeV" not in screen:
        continue
    valid = np.isfinite(screen["spec_MeV"]) & np.isfinite(screen["spec_MeV_per_mm"])
    if np.any(valid):
        ax.plot(screen["spec_MeV"][valid], screen["spec_MeV_per_mm"][valid], label=screen["label"])
ax.set(xlabel="Kinetic energy [MeV]", ylabel="Dispersion [MeV/mm]", title="Energy dispersion")
ax.legend()
ax.grid(True, alpha=0.25)
plt.show()
